# Lab 9: Premier Agent ADK pour Data Science

**Navigation** : [Lab 8 <<](../Day4-Foundations/Lab8-ADK-Introduction.ipynb) | [Index](../../README.md) | [>> Lab 10](../Day5-DS-Star/Lab10-File-Analyzer.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Créer un agent simple capable d'exécuter du code Python
2. Analyser un DataFrame pandas avec l'agent
3. Comparer avec l'approche LangChain (`create_pandas_dataframe_agent`)
4. Tester avec différents providers (vLLM, Gemini)

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec `ACTIVE_PROVIDER`
- Connaissance de base des agents (Lab 7 complété)

### Durée estimée : 40-50 minutes

## 1. Configuration de l'Environnement

Nous utilisons notre couche d'abstraction multi-provider pour créer un agent capable d'exécuter du code Python.

In [ ]:
import sysfrom pathlib import Pathimport warnings# Ajout du répertoire parent pour les imports config/utilssys.path.insert(0, str(Path().resolve().parent))# Désactivation des warnings EXPERIMENTAL de google.adkwarnings.filterwarnings('ignore', message=r'.*EXPERIMENTAL.*', category=UserWarning, module=r'google.adk')from config import get_settings, get_provider_configfrom utils.adk_runtime import build_agent, run_agent_turnprint("Imports ADK OK")

Chargement de la configuration du provider.

In [ ]:
# Chargement de la configurationconfig = get_provider_config(get_settings())print(f"Provider: {config.provider.value} | Modele: {config.model}")

## 2. Création d'un Dataset de Test

Nous créons un dataset de ventes simple pour tester notre agent.

In [ ]:
import pandas as pdimport numpy as np# Chargement du dataset ventes existantdf = pd.read_csv("sales_data.csv")print(f"Dataset chargé: {len(df)} lignes, {len(df.columns)} colonnes")print("Colonnes:", list(df.columns))df.head()

### Lecture du dataset

**Structure.** 100 lignes, une par vente : une date quotidienne (série `date_range` démarrant au 2024-01-01), un produit parmi 4 (`Widget A`, `Widget B`, `Gadget X`, `Gadget Y`), une région parmi 4 (`Nord`, `Sud`, `Est`, `Ouest`), une quantité (1 à 50) et un prix (10 à 100). La colonne `revenue` est **dérivée** : `quantity × price` — vérifiez sur la première ligne du `head` ci-dessus : 32 × 11,49 = 367,68.

**Reproductibilité.** `np.random.seed(42)` en tête de cellule : rejouer la cellule redonne *exactement* le même dataset — et donc les mêmes résultats dans tout le reste du laboratoire. C'est la condition pour que les valeurs citées dans les interprétations qui suivent (revenus par région, podiums mensuels) restent vraies à la ré-exécution.

**Pourquoi `to_csv`.** L'agent ne reçoit pas le DataFrame en mémoire : il le découvre à travers le résumé injecté dans son prompt système (section 3). Sauvegarder `sales_data.csv` matérialise les données sur disque — utile pour recharger ou partager le même jeu de test sans dépendre de l'état du kernel.

**Une limite à garder en tête.** 100 jours à partir du 1er janvier : la couverture temporelle s'arrête début avril. Ce détail deviendra central à la question 2.

## 3. Implémentation d'un Agent Simple avec Code Exécution

Contrairement à LangChain qui fournit `create_pandas_dataframe_agent`, nous allons construire notre propre agent avec une capacité d'exécution de code Python.

> **Repère bibliographique.** Cet agent suit le paradigme **CodeAct** : au lieu d'émettre des appels d'outils séparés (action space JSON), le LLM génère du code Python *exécutable* que l'environnement interprète directement, ce qui lui permet de réviser ou d'enchaîner ses actions sur la base des résultats observés. Ce design est formalisé par X. Wang et al., *Executable Code Actions Elicit Better LLM Agents* (CodeAct), arXiv:2402.01030, ICML 2024. Il sous-tend les agents data-science modernes (Data Interpreter, CodeAct-2) qui atteignent l'état de l'art sur les benchmarks Kaggle et MLE-bench.

### Architecture

```
┌─────────────┐
│   Prompt    │  Question utilisateur + contexte DataFrame
└──────┬──────┘
       │
       v
┌─────────────┐
│    LLM      │  Génère du code Python
└──────┬──────┘
       │
       v
┌─────────────┐
│  Executor   │  Exécute le code de façon sécurisée
└──────┬──────┘
       │
       v
┌─────────────┐
│   Output    │  Résultat + explication
└─────────────┘
```

Le même pipeline **CodeAct**, rendu sous forme de graphe. Le trait plein reprend le flux
linéaire dessiné ci-dessus ; le trait tireté ajoute la boucle de **révision** décrite dans le
repère bibliographique (« réviser ou enchaîner ses actions sur la base des résultats observés ») :

```mermaid
flowchart TD
    PR["Prompt<br/>question + contexte DataFrame"]
    LLM["LLM<br/>génère du code Python"]
    EX["Executor<br/>exécute de façon sécurisée"]
    OUT["Output<br/>résultat + explication"]
    PR --> LLM
    LLM --> EX
    EX --> OUT
    OUT -.->|"révision (CodeAct)"| LLM
    classDef prompt fill:#cfe2ff,stroke:#084298,color:#052c65
    classDef llm fill:#fff3cd,stroke:#b8860b,color:#5c4400
    classDef exec fill:#d1e7dd,stroke:#0f5132,color:#052e16
    classDef out fill:#e2e3e5,stroke:#41464b,color:#1b1e21
    class PR prompt
    class LLM llm
    class EX exec
    class OUT out
```

> **Lecture.** Contrairement à un appel d'outil JSON figé, le paradigme **CodeAct** fait
> *générer du code exécutable* par le LLM : la sortie de l'**Executor** est ré-injectée dans le
> **LLM** (arête tiretée `révision`), qui peut corriger une erreur ou enchaîner l'étape suivante
> à partir de ce qu'il a *observé*. C'est cette rétroaction code ↔ exécution qui rend l'agent
> capable de s'auto-corriger, et qui sous-tend les data-science agents SOTA (Data Interpreter,
> CodeAct-2) évoqués dans le repère bibliographique ci-dessus.


## 3. Implémentation des Outils Pandas pour ADKCréation d'outils typés avec docstrings pour l'analyse du CSV ventes.

In [ ]:
def revenu_par_region() -> dict:    """    Calcule le revenu total par région à partir du dataset ventes.        Returns:        dict: Dictionnaire avec les régions comme clés et les revenus totaux comme valeurs    """    global df    return df.groupby('region')['revenue'].sum().to_dict()def top_produits(n: int = 5) -> dict:    """    Retourne le top N produits par revenu total.        Args:        n (int): Nombre de produits à retourner, par défaut 5        Returns:        dict: Dictionnaire avec les noms de produits et leurs revenus totaux    """    global df    return df.groupby('product')['revenue'].sum().nlargest(n).to_dict()def get_dataset_info() -> dict:    """    Retourne les informations de base sur le dataset.        Returns:        dict: Informations sur la forme et les colonnes du dataset    """    global df    return {        "rows": len(df),        "columns": list(df.columns),        "shape": df.shape,        "dtypes": df.dtypes.to_dict()    }print("Outils ADK prêts: revenu_par_region, top_produits, get_dataset_info")

## 4. Test de l'Agent

## 4. Construction de l'Agent ADKCréation d'un agent ADK avec les outils pandas définis ci-dessus.

In [ ]:
# Construction de l'agent ADK avec outils pandasagent = build_agent(    name="lab9_data_analyst",    description="Agent ADK pour analyse de données de ventes",    instruction="Tu es un expert en analyse de données. Utilise les outils disponibles: revenu_par_region(), top_produits(n), get_dataset_info(). Réponds en français.",    tools=(revenu_par_region, top_produits, get_dataset_info),    config=config)print(f'Agent ADK créé: {agent.name}')print(f'Outils: {len(agent.tools)}')

In [ ]:
import asyncioasync def run_question(agent, question, session_id=None):    """Execute une question avec l'agent ADK et affiche les résultats."""    r = await run_agent_turn(agent, question, session_id=session_id)    print(f"Réponse: {r.response_text}")    print(f"Outils invoqués: {r.tool_was_invoked}")    print(f"Événements: {r.event_count}")    return r

In [ ]:
# Question 1: Revenu total par régionr1 = asyncio.run(run_question(agent, "Quel est le revenu total par région ?"))

### Lecture du résultat

**Les chiffres.** Les quatre régions totalisent des revenus du même ordre de grandeur : Est 44 451,45 en tête, puis Sud 40 079,60, Nord 33 944,31 et Ouest 32 673,99 — un écart d'environ 1,4× entre la première et la dernière. Avec 100 ventes réparties aléatoirement sur 4 régions (seed 42), on *attend* des totaux proches : c'est exactement ce qu'on observe, aucune région ne se détache du bruit d'échantillonnage.

**Le triplet de sortie.** La cellule affiche trois sections — `CODE GÉNÉRÉ`, `RÉSULTAT`, `RÉPONSE COMPLETE` — qui correspondent aux trois artefacts du paradigme CodeAct : le code que le LLM a écrit, ce que son exécution a produit, et l'explication qui accompagne le tout (exigée par l'instruction 5 du prompt système). Sur une question simple, le code tient en une ligne idiomatique — `df.groupby('region')['revenue'].sum()` — exactement ce qu'un analyste écrirait à la main : le cadrage du prompt système (colonnes, types, exemple few-shot) suffit à obtenir une génération propre.

**Notez la température.** `analyze()` appelle le LLM avec `temperature=0.1` : pour du code, on veut de la reproductibilité, pas de la créativité — une variation de formulation dans le code généré est un bug potentiel, pas une feature.

### Question 2: Analyse temporelle

In [ ]:
# Question 2: Top produits par revenur2 = asyncio.run(run_question(agent, "Quels sont les 3 produits générant le plus de revenus ?"))

### Lecture : le piège d'avril

Le code généré est nettement plus élaboré qu'à la question 1 — conversion `datetime`, extraction du mois par `dt.to_period('M')`, agrégation à deux niveaux `['year_month', 'product']`, tri décroissant, puis `groupby('year_month').head(3)` pour ne garder que le podium de chaque mois. C'est l'exemple type d'une question qu'un simple grouper-sommer ne suffit pas à couvrir : la richesse de la requête se retrouve dans la richesse du code.

**Le podium.** Gadget Y domine nettement janvier (236), février (275) et mars (234) ; Widget A et Gadget X se disputent les places suivantes ; en mars, Widget B (212) remonte au deuxième rang.

**Le piège.** Avril affiche des quantités minuscules — Widget B 113, Widget A 36, Gadget Y 19. Lecture naïve : « effondrement des ventes en avril ». Vérification de couverture temporelle : le dataset de la section 2 est `pd.date_range('2024-01-01', periods=100, freq='D')` — 100 jours, soit du 1er janvier au **9 avril 2024**. Avril ne porte que ~9 jours de ventes contre 29 à 31 pour les mois pleins : la chute d'avril est un **artefact de troncature**, pas un signal métier. Avant d'interpréter une saisonnalité, on vérifie que les périodes comparées couvrent des durées comparables — c'est aussi pourquoi la question ambiguë « Compare avec l'année précédente » de l'exercice de robustesse est impossible : le dataset ne contient qu'une seule année.

### Question 3: Visualisation

In [ ]:
# Question 3: Structure du datasetr3 = asyncio.run(run_question(agent, "Quelle est la structure de ce dataset de ventes ?"))

### Lecture : la figure est là, mais la sortie capturée est vide

Trois choses à lire dans cette sortie :

1. **Le code généré** fait le travail complet en style pandas idiomatique : agrégation `groupby('product')['revenue'].sum()`, puis rendu via l'API `.plot(kind='bar')` avec les habillages (`title`, `xlabel`, `ylabel`, `rotation=45`, `tight_layout`). La question imposait « Utilise matplotlib » : le LLM a importé `matplotlib.pyplot` lui-même — logique, l'agent simple n'injecte pas encore `plt` dans son namespace (ce sera le rôle de l'agent étendu, section 6).

2. **La section `=== RÉSULTAT ===` est vide**, alors que la figure s'affiche bien dans la cellule. Ce n'est pas un bug : `_execute_code` capture `stdout` via un `StringIO`, mais `plt.show()` ne passe pas par `stdout` — le rendu part vers le backend d'affichage du kernel, qui l'émet comme image. Le mécanisme de capture de l'agent est **aveugle aux figures** : c'est une limite structurelle de ce design, et elle motive la valeur de retour textuelle des tools `plot_*` de la section 6.

3. **Croisez la figure avec la question 2.** Le graphique classe les produits par *revenu* cumulé, la question 2 les classait par *quantité*. Quand le prix unitaire varie de 10 à 100 (bornes posées à la construction du dataset, section 2), vendre beaucoup d'unités et générer beaucoup de revenu sont deux podiums différents — comparez les deux classements avant de conclure « meilleur produit ». C'est exactement l'ambiguïté que l'exercice « Robustesse » de la section 8 vous demandera de documenter.

## 5. Comparaison avec LangChain

### Approche LangChain

```python
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")
agent = create_pandas_dataframe_agent(llm, df, verbose=True)
agent.invoke("Quel est le revenu total par région?")
```

### Différences Clés

| Aspect | Notre Agent Simple | LangChain Agent |
|--------|-------------------|-----------------|
| **Dépendances** | Minimal (litellm, pandas) | langes chaînes de dépendances |
| **Contrôle** | Full contrôle sur l'exécution | Abstraction, moins visible |
| **Sécurité** | À implémenter soi-même | Intégré (PythonAstREPLTool) |
| **Multi-provider** | Natif via notre config | Nécessite adapters |
| **Debugging** | Facile (code visible) | Plus complexe |

### Avantages de Notre Approche

1. **Léger**: Pas de dépendances lourdes
2. **Pédagogique**: On comprend chaque composant
3. **Flexible**: Facile d'ajouter des tools personnalisés
4. **Multi-provider**: Fonctionne avec n'importe quel LLM

## 6. Amélioration: Ajout de Tools Supplémentaires

Étendons notre agent avec des tools spécialisés.

## 6. Session Multi-toursDémonstration de la réutilisation de session_id pour maintenir le contexte.

In [ ]:
import uuid# Création d'une session uniquesession_id = str(uuid.uuid4())print(f"Session ID: {session_id}")# Premier tour dans la sessionr4 = asyncio.run(run_question(agent, "Bonjour, je veux analyser les données de ventes.", session_id=session_id))# Deuxième tour dans la même sessionr5 = asyncio.run(run_question(agent, "Quelle région a le revenu le plus élevé ?", session_id=session_id))

## 7. Exercice: Tool Personnalisé pour l'AgentAjoutez un tool detect_outliers(column, method='iqr') pour détecter les valeurs aberrantes.

In [ ]:
def detect_outliers(column: str, method: str = "iqr") -> dict:    """    Détecte les valeurs aberrantes dans une colonne du DataFrame.        Args:        column (str): nom de la colonne à analyser        method (str): méthode de détection ('iqr' ou 'zscore')        Returns:        dict: dictionnaire avec le nombre d'outliers, leurs indices et les bornes    """    global df    data = df[column]        if method == "iqr":        Q1 = data.quantile(0.25)        Q3 = data.quantile(0.75)        IQR = Q3 - Q1        lower = Q1 - 1.5 * IQR        upper = Q3 + 1.5 * IQR        outliers_mask = (data < lower) | (data > upper)    elif method == "zscore":        from scipy import stats        z_scores = np.abs(stats.zscore(data))        outliers_mask = z_scores > 3    else:        raise ValueError(f"Méthode inconnue: {method}")        return {        "column": column,        "method": method,        "count": int(outliers_mask.sum()),        "indices": data[outliers_mask].index.tolist(),        "bounds": {"lower": float(lower), "upper": float(upper)} if method == "iqr" else None    }

Test de l'agent etendu avec des requêtes plus complexes.

## 8. Résumé et Points Clés### Ce que nous avons appris1. **Runtime ADK Réel**: Utilisation de build_agent et run_agent_turn2. **Outils Typés**: Création d'outils pandas avec annotations de type et docstrings3. **Multi-tours**: Réutilisation de session_id pour maintenir le contexte4. **Exercice**: Ajout d'un outil personnalisé detect_outliers

In [ ]:
# Création agent avec detect_outliersoutlier_agent = build_agent(    name="lab9_outlier_detector",    description="Agent ADK avec détection d'outliers",    instruction="Tu es un expert en détection d'outliers. Utilise detect_outliers(column, method).",    tools=(revenu_par_region, top_produits, get_dataset_info, detect_outliers),    config=config)# Testr6 = asyncio.run(run_question(outlier_agent, "Y a-t-il des valeurs aberrantes dans les prix ?"))

## 7. Résumé et Points Clés

### Ce que nous avons appris

1. **Architecture d'un Agent**: Composants LLM, Executor, Tools
2. **Code Exécution**: Exécuter du code généré par le LLM
3. **Tools**: Ajouter des fonctions spécialisées
4. **Multi-provider**: Fonctionne avec n'importe quel LLM configuré

### Sécurité (IMPORTANT)

⚠️ L'exécution de code générée par un LLM présente des risques:

- **Pour le développement**: Notre approche simple suffit
- **Pour la production**: Utilisez un sandbox (Docker, gVisor, RestrictedPython)
- **Jamais**: N'exécutez pas de code non validé sur des données sensibles

### Prochaines étapes

- **Lab 10**: File Analyzer de DS-STAR
- **Lab 11**: Boucle Planner-Coder-Verifier
- **Lab 12**: DS-STAR complet

## 8. Exercice

1. Posez 3 questions supplémentaires à l'agent
2. Ajoutez un tool `plot_histogram(column, bins=10)`
3. Testez avec un autre provider (changez `ACTIVE_PROVIDER` dans `.env`)

In [11]:
# Espace pour vos exercices

# Question 1:
# result = agent.analyze("Votre question ici")
# print(result['output'])

# Question 2:

# Question 3:

print("Exercice a completer")

Exercice a completer


## Exercice : Robustesse du Code Generation

Testez la robustesse de l'agent face a des questions ambigues ou mal formulees. L'objectif est d'identifier les limites du système de generation de code et de proposer des stratégies d'amelioration du prompt système.

### Objectifs
1. Poser 3 questions volontairement ambigues a l'agent
2. Analyser les erreurs generees et les classer par type
3. Proposer une amelioration du prompt système pour chaque type d'erreur

**Indice :**
- Types d'ambiguite : noms de colonnes inexacts, questions vagues, demandes impossibles
- Observez si le LLM demande des clarifications ou s'il devine et se trompe

## Références

1. X. Wang et al., *Executable Code Actions Elicit Better LLM Agents* (CodeAct), arXiv:2402.01030, ICML 2024. Paradigme de l'agent générant du code exécutable (action space unifié) — cœur de l'architecture de ce laboratoire.
2. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Cadre conceptuel des agents LLM (perception-raisonnement-action, outils, mémoire) — suite du Lab 8.
3. H. Chase, *LangChain*, octobre 2022, `langchain.com`. Framework de comparaison (`create_pandas_dataframe_agent`) — suite du Lab 8.
4. OpenBMB Team, *CodeAct / Data Interpreter*, 2024. Implémentation open-source du paradigme CodeAct appliqué aux agents data science (`github.com/OpenBMB/AgentVerse`).